In [ ]:
### Install requirements
from pathlib import Path

REPOSITORY = "https://github.com/lmingari/olot-course.git"
PROJECT_FILE = "pyproject.toml"
BRANCH = "extra"

if not Path(PROJECT_FILE).exists():
    !git clone -q -b {BRANCH} {REPOSITORY}
    %cd olot-course
    !pip install -q -e ".[data,ml]"

In [ ]:
### GPU are required here!
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "CUDA is required. In Colab: Runtime > Change runtime type > GPU."
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# 4. Generative AI
---

## Problem statement

Let $\{x_i\}_{i=1}^{N}$ be a dataset of independent observations assumed to be drawn from an unknown probability distribution:

$$x_i \sim p_{\text{data}}, \qquad i = 1, \dots, N.$$

* The objective of generative modeling is to produce **new samples** $x \sim p_{\text{data}}$
* New samples should be statistically similar to the observations used to train the model
* Can a model learn the probability distribution underlying the data?

## Challenges

* $p_{\text{data}}$ is not available in closed form, it cannot cannot be evaluated directly
* We can only access $p_{\text{data}}$ through a finite set of samples: no direct procedure for drawing new samples from the distribution
* Scientific data such as images and simulation fields are high-dimensional, complex, and highly structured

## Generative models

**Generative models** learn a representation of the data distribution from available samples and provide a practical way to generate new ones

Instead of sampling from $p_{\text{data}}$ directly, we introduce a reference distribution $p_{\text{simple}}$ from which sampling is straightforward, for example a standard Gaussian:

Instead of sampling directly from the unknown distribution $p_{\text{data}}$, we introduce a simple **reference distribution** $p_{\text{simple}}$ from which sampling is straightforward. For example:

$$z \sim p_{\text{simple}} = \mathcal{N}(0, I).$$

The goal is then to learn a transformation that maps samples from the simple distribution into samples that follow the data distribution:

$$
z \sim p_{\text{simple}}
\quad\longmapsto\quad
x \sim p_{\text{data}}.
$$

> **Idea:** \
> generate complex data by transforming simple random samples

## &#128221; Example 1: The variational autoencoder (VAE)

A traditional autoencoder learns two functions, an **encoder** that maps data to a latent variable and a **decoder** that maps it back:

$$z = f_\phi(x), \qquad \hat{x} = g_\theta(z).$$

A **variational autoencoder (VAE)** makes both maps probabilistic. 
Each neural network provides the mean of a Gaussian conditional distribution:

$$q_\phi(z \mid x) = \mathcal{N}\big(z;\, f_\phi(x),\, \sigma_\phi^2(x) I\big)$$
$$p_\theta(x \mid z) = \mathcal{N}\big(x;\, g_\theta(z),\, \sigma^2 I\big)$$

We could generate a sample by setting $z \sim p(z) = \mathcal{N}(0, I)$ and sampling $x \sim p_\theta(\cdot \mid z)$ from the decoder, resulting in the distribution:

$$p_\theta(x) = \int p_\theta(x \mid z)\, p(z)\, dz$$

The goal is for $p_\theta(x)$ to **approximate** $p_{\text{data}}(x)$.

### VAE training

Ideally, we want to minimize the KL divergence between the data distribution and the model distribution:

$$D_{\mathrm{KL}}\big(p_{\text{data}} \,\|\, p_\theta\big)$$

The KL divergence is a standard measure of dissimilarity between distributions. It is zero if and only if the two distributions are equal, and positive otherwise. 

However, $p_\theta(x)$ is intractable, so in practice the loss function is defined in terms of the **joint distributions**:

$$\mathcal{L}_{VAE}(\theta, \phi) = D_{\mathrm{KL}}\big(q_\phi(x, z) \,\|\, p_\theta(x, z)\big)$$

where:

$$
q_\phi(x, z) = p_{\text{data}}(x)\, q_\phi(z \mid x)
$$

$$
p_\theta(x, z) = p(z)\, p_\theta(x \mid z)
$$

This is an upper bound on $D_{\mathrm{KL}}\big(p_{\text{data}} \,\|\, p_\theta\big)$, so minimizing it pushes $p_\theta$ toward $p_{\text{data}}$.

### Application to volcanic plumes

| VAE-generated samples for a volcanic plume |
| -----------------------               |
| <img src="figs/vae-sampling.png" width=600 />  |
| __Figure 1:__ Garachico eruption |

| Histograms for a VAE-generated ensemble with 5000 samples |
| -----------------------               |
| <img src="figs/vae-histograms.png" width=600 />  |
| __Figure 2:__ Comparison between VAE-generated ensemble and simulated ensembles |

## &#128221; Example 2: Generation as a dynamical system

The transformation from a simple distribution to the data distribution can also be described as a dynamical system.

Instead of mapping $z$ directly to $x$, we define a continuous trajectory $x_t$ governed by an ordinary differential equation:

$$
\frac{d x_t}{dt} = u_\theta(x_t,t),
\qquad t \in [0,1].
$$

The trajectory starts from a sample of the simple distribution:

$$
x_0 = z \sim p_{\text{simple}},
$$

and evolves according to the learned vector field $u_\theta$ until $t=1$:

$$
x_1 \sim p_{\text{data}}.
$$

The learned vector $u_\theta(x,t)$ determines the direction and speed of the transformation at each point in the state space.

### Flow vs Diffusion

The dynamical-system view leads to two closely related approaches for generative modelling:

| Model| Dynamics |
| :--- | --- |
| **Flow models (ODE):**      | $$\frac{dx_t}{dt} = u_\theta(x_t, t), \qquad x_0 = z \sim p_{\text{simple}}$$ | 
| **Diffusion models (SDE):** | $$dx_t = u_\theta(x_t, t)\,dt + \sigma_t\,dW_t, \qquad x_0 = z \sim p_{\text{simple}}$$ |

- **Flow models**: the particle follows the learned vector field deterministically
- **Diffusion models**: the particle follows the vector field while also being perturbed by random noise.
  Here, $W_t$ is a Brownian motion and $\sigma_t$ controls the noise strength.

| Flow vs Diffusion |
| ----------------- |
| <img src="figs/flow-vs-difussion.png" width=600 />  |
| __Figure 3:__ Figure extracted from [Steven Gong, Flow Matching](https://stevengong.co/notes/Flow-Matching) |

### Flow matching

Take a data sample $x_1 \sim p_{\text{data}}$ and corrupt it with noise $z \sim \mathcal{N}(0, I)$ by an amount that depends on time $t \in [0, 1]$:

$$x_t = \alpha_t\, z + \beta_t\, x_1,$$

with $\alpha_0 = 1,\ \beta_0 = 0$ (pure noise) and $\alpha_1 = 0,\ \beta_1 = 1$ (clean data). We use the simplest choice, $\alpha_t = 1 - t$ and $\beta_t = t$.

The network $u_\theta(x_t, t)$ learns to **denoise**: at every noise level, it predicts the direction in which $x_t$ moves along this path,

$$\frac{dx_t}{dt} = \dot\alpha_t\, z + \dot\beta_t\, x_1 = x_1 - z.$$

This is a regression problem with the loss

$$\mathcal{L}(\theta) = \mathbb{E}_{z,\,x_1,\,t}\,\big\| u_\theta(x_t, t) - (x_1 - z) \big\|^2.$$

## &#128221; Example 3: Sampling from a pre-trained flow model

Training a generative model can be computationally demanding. Instead, we will use a pre-trained flow model that has already learned the distribution of FALL3D simulations.

### Loading the model

The trained model is stored in a `checkpoint`, which contains both the model configuration and its learned parameters:

In [ ]:
import torch
from course.model import FlowUNet

checkpoint = torch.load("flow_model.pt", map_location=device)

model = FlowUNet(**checkpoint["model_config"])
model.load_state_dict(checkpoint["model_state"])
model = model.to(device).eval()

H, W = checkpoint['shape'] # grid size used in training

### Sampling from the flow

We start from a random sample $x_0 = z \sim \mathcal{N}(0,I)$ and integrate the learned ODE from $t=0$ to $t=1$ using Euler's method:

$$
x_{t+\Delta t} = x_t + \Delta t\,u_\theta(x_t,t)
$$

In [ ]:
steps = 50
dt = 1.0 / steps
save_at = [0, 5, 10, 20, 30, 40, 50]               # steps at which we store x_t

x = torch.randn(1, 1, H, W, device=device)         # z ~ N(0, I)
frames = {0: x.cpu()}

with torch.no_grad():
    for k in range(steps):
        t = torch.full((1,), k * dt, device=device)
        x = x + dt * model(x, t)
        if k + 1 in save_at:
            frames[k + 1] = x.cpu()

### Visualising the generation process
The stored frames show how the initial random field evolves through the flow from $t=0$ to $t=1$.

In [ ]:
import matplotlib.pyplot as plt

conf_plt = dict(vmin=0, vmax=5, origin="lower")

fig, axes = plt.subplots(1, len(save_at), figsize=(2.5 * len(save_at), 2.8))
for ax, k in zip(axes, save_at):
    ax.imshow(frames[k][0, 0], **conf_plt)
    ax.set_title(f"t = {k / steps:.2f}")
    ax.axis("off")

### Trainer object

In [ ]:
import torch
import torch.nn as nn
from course.model import FlowUNet
from course.training import FlowMatchingTrainer

model_config = dict(n_levels=4)
model = FlowUNet(**model_config)

trainer = FlowMatchingTrainer(
    model     = model,
    criterion = nn.MSELoss(),
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3),
)

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"device: {trainer.device} | parameters: {n_params / 1e6:.2f} M")

In [ ]:
from torchinfo import summary

summary(model, input_size=[(1, 1, H, W), (1,)], device=device, depth=3)

### Get raw data

In [ ]:
### Get data
from course.helper import download_file

fname_train = download_file("https://saco.csic.es/public.php/dav/files/tsK7yQGM7BLaWpR")
fname_val   = download_file("https://saco.csic.es/public.php/dav/files/6BfKXP8Gwm6sdeJ")

In [ ]:
import xarray as xr

da_train = xr.open_dataset(fname_train)["SO2_col_mass"]
da_val   = xr.open_dataset(fname_val)["SO2_col_mass"]

### Dataset and data loader

In [ ]:
from torch.utils.data import DataLoader
from course.dataset import SimulationDataset, LogTransform

coarsen = (4, 4)
transform = LogTransform()

# Datasets
train_dataset = SimulationDataset(da_train, transform, coarsen=coarsen)
val_dataset   = SimulationDataset(da_val,   transform, coarsen=coarsen)

# Dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=16, 
    pin_memory=torch.cuda.is_available(),  # faster CPU -> GPU copies
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    pin_memory=torch.cuda.is_available(),  # faster CPU -> GPU copies
    shuffle=False
)

In [ ]:
C, H, W = train_dataset[0].shape

print("train item shape:", C, H, W)
print("train std:", float(train_dataset.data.std()))           # should be close to 1

In [ ]:
batch = next(iter(train_loader))
print("shape:", tuple(batch.shape), "| dtype:", batch.dtype)   # expected: (B, 1, H, W), torch.float32
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

### Training loop

In [ ]:
history = trainer.fit(train_loader, val_loader, epochs=200)

In [ ]:
history.plot(
    x="epoch",
    y=["train_loss", "val_loss"],
)

In [ ]:
checkpoint = {
    "model_state": model.state_dict(),
    "model_config": model_config,
    "transform": {"factor": 1, "scale": 1},
    "coarsen": coarsen,
    "shape": (H,W),
}
torch.save(checkpoint, "flow_model.pt")

### Generating multiple samples
We can apply the same flow to multiple independent noise samples at once. Each sample follows its own trajectory through the learned vector field.

In [ ]:
nsamples = 6
steps = 50
dt = 1.0 / steps

x = torch.randn(nsamples, 1, H, W, device=device)         # z ~ N(0, I), (nsamples, C, H, W)

with torch.no_grad():
    for k in range(steps):
        t = torch.full((nsamples,), k * dt, device=device)    # current time, (nsamples,)
        x = x + dt * model(x, t)                              # Euler step

samples = x.cpu()  

The result is an ensemble of nsamples new fields, generated from independent samples of the Gaussian distribution.

To assess the generated samples, we can compare them with fields from the validation dataset:

In [ ]:
real = val_dataset.data[:nsamples]                            # (nsamples, 1, H, W), transformed space

In [ ]:
fig, axes = plt.subplots(2, nsamples, figsize=(2.5 * nsamples, 5))

for i in range(nsamples):
    axes[0, i].imshow(real[i, 0], **conf_plt)
    axes[0, i].set_title("FALL3D")
    axes[0, i].axis("off")

    axes[1, i].imshow(samples[i, 0], **conf_plt)
    axes[1, i].set_title("Generated")
    axes[1, i].axis("off")

plt.tight_layout()